<a href="https://colab.research.google.com/github/RozieKii/localTest/blob/main/evaluations_testing_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.2: Evaluations - Testing Your AI

## Key Insight: How to write 'Unit Tests' for an AI

You need to TEST your digital twin like you test code!

## What You'll Build
- Test datasets for your AI
- Evaluation metrics (accuracy, hallucination)
- Automated testing framework
- Quality assurance for your twin

In [1]:
from typing import List, Dict, Any
import json

print('Week 3.2: Evaluations & Testing')
print('Goal: Ensure your digital twin works correctly!')

Week 3.2: Evaluations & Testing
Goal: Ensure your digital twin works correctly!


## Part 1: Creating Test Cases

Test cases have: input, expected output, pass criteria.

In [2]:

class TestCase:
    """Single test case for AI evaluation."""

    def __init__(
        self,
        input_text: str,
        expected_output: str,
        criteria: str = 'exact_match'
    ):
        self.input = input_text
        self.expected = expected_output
        self.criteria = criteria

    def evaluate(self, actual_output: str) -> Dict[str, Any]:
        """Evaluate if test passed."""

        if self.criteria == 'exact_match':
            passed = actual_output.strip() == self.expected.strip()

        elif self.criteria == 'contains':
            passed = self.expected.lower() in actual_output.lower()

        elif self.criteria == 'keyword':
            keywords = self.expected.split()
            passed = all(
                kw.lower() in actual_output.lower()
                for kw in keywords
            )

        else:
            passed = False

        return {
            'passed': passed,
            'input': self.input,
            'expected': self.expected,
            'actual': actual_output,
            'criteria': self.criteria
        }


# Example test cases
test_cases = [
    TestCase(
        "What is 2 + 2?",
        "4",
        "contains"
    ),
    TestCase(
        "Who created you?",
        "I am a digital twin",
        "contains"
    ),
    TestCase(
        "What's the weather?",
        "I don't have access to current weather",
        "contains"
    )
]

print(f'Created {len(test_cases)} test cases')


Created 3 test cases


## Part 2: Evaluation Metrics

Measure quality of AI responses.

In [3]:

class EvaluationMetrics:
    """Calculate metrics for AI outputs."""

    @staticmethod
    def accuracy(results: List[Dict]) -> float:
        """Calculate accuracy (proportion passed)."""
        if not results:
            return 0.0

        passed = sum(1 for r in results if r['passed'])
        return passed / len(results)

    @staticmethod
    def check_hallucination(output: str, context: str) -> bool:
        """Roughly flag outputs with many words absent from context."""
        output_words = set(output.lower().split())
        context_words = set(context.lower().split())

        unique_words = output_words - context_words
        hallucination_ratio = (
            len(unique_words) / len(output_words)
            if output_words else 0
        )

        return hallucination_ratio > 0.7

    @staticmethod
    def relevance_score(query: str, response: str) -> float:
        """Calculate word-based Jaccard similarity (0–1)."""
        query_words = set(query.lower().split())
        response_words = set(response.lower().split())

        intersection = query_words & response_words
        union = query_words | response_words

        return len(intersection) / len(union) if union else 0.0


metrics = EvaluationMetrics()

# Example
query = "What is machine learning?"
response = "Machine learning is a type of AI that learns from data"

print(f'Relevance: {metrics.relevance_score(query, response):.2f}')


Relevance: 0.15


## Part 3: Test Suite

Run all tests and report results.

In [4]:

class AITestSuite:
    """Test suite for AI systems."""

    def __init__(self, name: str):
        self.name = name
        self.test_cases = []
        self.results = []

    def add_test(self, test_case: TestCase):
        """Add a test case."""
        self.test_cases.append(test_case)

    def run_tests(self, ai_function) -> Dict[str, Any]:
        """Run all tests and calculate metrics."""
        self.results = []

        for test in self.test_cases:
            actual = ai_function(test.input)
            result = test.evaluate(actual)
            self.results.append(result)

        accuracy = EvaluationMetrics.accuracy(self.results)

        return {
            'total_tests': len(self.test_cases),
            'passed': sum(1 for r in self.results if r['passed']),
            'failed': sum(1 for r in self.results if not r['passed']),
            'accuracy': accuracy
        }

    def report(self):
        """Print test report."""
        print(f'\n{"=" * 60}')
        print(f'Test Suite: {self.name}')
        print(f'{"=" * 60}')

        for i, result in enumerate(self.results, 1):
            status = '✓ PASS' if result['passed'] else '✗ FAIL'
            print(f'\nTest {i}: {status}')
            print(f"  Input: {result['input']}")
            print(f"  Expected: {result['expected']}")
            print(f"  Actual: {result['actual'][:50]}...")

        accuracy = EvaluationMetrics.accuracy(self.results)
        passed = sum(1 for r in self.results if r['passed'])

        print(f"\n{'-' * 60}")
        print(f"Accuracy: {accuracy * 100:.1f}%")
        print(f"Passed: {passed}/{len(self.results)}")


# Create the test suite and add our existing tests
suite = AITestSuite('Digital Twin Tests')

for tc in test_cases:
    suite.add_test(tc)

print(f'\nTest suite ready with {len(test_cases)} tests')



Test suite ready with 3 tests


## Part 4: Testing Your Digital Twin

Apply tests to your AI.

In [7]:

import re
import ast
import operator

def mock_ai(query: str) -> str:
    """Improved demo twin with basic arithmetic and safe fallback."""

    if not isinstance(query, str) or not query.strip():
        return "I don't understand"

    text = query.lower()

    if "weather" in text:
        return "I don't have access to current weather data"

    if "created" in text:
        return "I am a digital twin assistant"

    # Find simple arithmetic expressions in a query
    match = re.search(r'(-?\d+(?:\.\d+)?)\s*([+*/-])\s*(-?\d+(?:\.\d+)?)', text)

    if match:
        a = float(match.group(1))
        op = match.group(2)
        b = float(match.group(3))

        operations = {
            '+': operator.add,
            '-': operator.sub,
            '*': operator.mul,
            '/': operator.truediv
        }

        if op == '/' and b == 0:
            return "Cannot divide by zero"

        result = operations[op](a, b)
        if result.is_integer():
            result = int(result)
        return f"The answer is {result}"

    return "I don't understand"


In [8]:
results = my_suite.run_tests(mock_ai)
my_suite.report()
print(f"\nOverall Accuracy: {results['accuracy'] * 100:.1f}%")


Test Suite: My Digital Twin Tests

Test 1: ✓ PASS
  Input: What is 2 + 2?
  Expected: 4
  Actual: The answer is 4...

Test 2: ✓ PASS
  Input: Who created you?
  Expected: I am a digital twin
  Actual: I am a digital twin assistant...

Test 3: ✓ PASS
  Input: What's the weather?
  Expected: I don't have access to current weather
  Actual: I don't have access to current weather data...

Test 4: ✓ PASS
  Input: What is 10 * 5?
  Expected: 50
  Actual: The answer is 50...

Test 5: ✓ PASS
  Input: Help me plan my goals
  Expected: I don't understand
  Actual: I don't understand...

Test 6: ✓ PASS
  Input: Save my task: study AI
  Expected: I don't understand
  Actual: I don't understand...

Test 7: ✓ PASS
  Input: Tell me an unknown fact
  Expected: I don't understand
  Actual: I don't understand...

Test 8: ✓ PASS
  Input: 
  Expected: I don't understand
  Actual: I don't understand...

Test 9: ✓ PASS
  Input: What is 100 / 4?
  Expected: 25
  Actual: The answer is 25...

Test 10: ✓ PASS


## Exercise 1: Create Your Test Suite

**Task**: Create 10 test cases for YOUR digital twin.

1. Cover different capabilities
2. Include edge cases
3. Test for hallucinations
4. Run the suite
5. Improve your twin based on failures

In [6]:

# Create our test suite
my_suite = AITestSuite('My Digital Twin Tests')

# Add 10 test cases covering capabilities and edge cases
my_tests = [
    TestCase("What is 2 + 2?", "4", "contains"),
    TestCase("Who created you?", "I am a digital twin", "contains"),
    TestCase("What's the weather?", "I don't have access to current weather", "contains"),
    TestCase("What is 10 * 5?", "50", "contains"),
    TestCase("Help me plan my goals", "I don't understand", "exact_match"),
    TestCase("Save my task: study AI", "I don't understand", "exact_match"),
    TestCase("Tell me an unknown fact", "I don't understand", "exact_match"),
    TestCase("", "I don't understand", "exact_match"),
    TestCase("What is 100 / 4?", "25", "contains"),
    TestCase("Tell me today's weather forecast", "I don't have access to current weather", "contains")
]

for test in my_tests:
    my_suite.add_test(test)

# Run the tests against the current mock AI
results = my_suite.run_tests(mock_ai)

# Display the report
my_suite.report()
print(f"\nOverall Accuracy: {results['accuracy'] * 100:.1f}%")



Test Suite: My Digital Twin Tests

Test 1: ✓ PASS
  Input: What is 2 + 2?
  Expected: 4
  Actual: The answer is 4...

Test 2: ✓ PASS
  Input: Who created you?
  Expected: I am a digital twin
  Actual: I am a digital twin assistant...

Test 3: ✓ PASS
  Input: What's the weather?
  Expected: I don't have access to current weather
  Actual: I don't have access to current weather data...

Test 4: ✗ FAIL
  Input: What is 10 * 5?
  Expected: 50
  Actual: I don't understand....

Test 5: ✗ FAIL
  Input: Help me plan my goals
  Expected: I don't understand
  Actual: I don't understand....

Test 6: ✗ FAIL
  Input: Save my task: study AI
  Expected: I don't understand
  Actual: I don't understand....

Test 7: ✗ FAIL
  Input: Tell me an unknown fact
  Expected: I don't understand
  Actual: I don't understand....

Test 8: ✗ FAIL
  Input: 
  Expected: I don't understand
  Actual: I don't understand....

Test 9: ✗ FAIL
  Input: What is 100 / 4?
  Expected: 25
  Actual: I don't understand....

Test 1

## Summary

### What We Learned:
- Test cases = input + expected + criteria
- Metrics: accuracy, hallucination, relevance
- Test suites automate quality checks
- Regular testing catches regressions

### Next: Week 3.3 - Guardrails & Safety